In [ ]:
"""
Portfolio management module for trading simulation.

This module defines the Portfolio class, which manages cash and positions,
enforcing leverage limits and providing methods to buy and sell products.

Author: Mathis Makarski
Date: 2025-11-02
"""

import logging
from pricing.Market import Market

#logger = logging.getLogger("local_eval")

class Portfolio():
    def __init__(self, cash: float, market: "Market", leverage_limit: float) -> None:
        self.cash: float = cash
        self.market: Market = market
        self.positions: dict[str, int] = {}  # key: product, value: quantity
        self.leverage_limit: float = leverage_limit  # max leverage allowed

    def _get_price(self, product: str) -> float:
        """Retrieve the last market price for a given product."""
        if product not in self.market.quotes:
            raise ValueError(f"No quote available for {product}")
        return self.market.quotes[product].get("price", None)
    
    def _get_timestamp(self, product) -> int:
        """Retrieve the current market timestamp for the product quote."""
        if product not in self.market.quotes:
            raise ValueError(f"No quote available for {product}")
        return self.market.quotes[product].get("timestep", None)

    def _gross_exposure(self) -> float:
        """Compute gross exposure = sum(|position| * price)"""
        total = 0.0
        for product, qty in self.positions.items():
            price = self._get_price(product)
            total += abs(qty) * price
        return total

    def _net_asset_value(self) -> float:
        """Compute portfolio net asset value = cash + sum(qty * price)"""
        value = self.cash
        for product, qty in self.positions.items():
            price = self._get_price(product)
            value += qty * price
        return value
    
    def _leverage(self) -> float:
        """Compute current leverage = gross exposure / net asset value"""
        gross = self._gross_exposure()
        net_value = self._net_asset_value()
        return gross / max(net_value, 1e-8)  # Avoid division by zero

    def _check_leverage(self, new_cash: float, new_positions: dict[str, int]) -> bool:
        """Check whether the new portfolio state respects leverage limits."""
        gross = sum(abs(qty) * self._get_price(p) for p, qty in new_positions.items())
        net_value = new_cash + sum(qty * self._get_price(p) for p, qty in new_positions.items())
        leverage = gross / max(net_value, 1e-8)
        return leverage <= self.leverage_limit

    def buy(self, product: str, quantity: int) -> bool:
        """Attempt to buy `quantity` units of `product`."""
        timestamp = self._get_timestamp(product)
        price = self._get_price(product)
        cost = price * quantity

        new_cash = self.cash - cost
        new_positions = self.positions.copy()
        new_positions[product] = new_positions.get(product, 0) + quantity

        if not self._check_leverage(new_cash, new_positions):
            #logger.warning(f"{timestamp} | Trade rejected: leverage limit exceeded.")
            return False

        self.cash = new_cash
        self.positions = new_positions
        #logger.info(f"{timestamp} | BOUGHT {quantity} {product} @ {price} | new cash={self.cash:.2f}")
        return True

    def sell(self, product: str, quantity: int) -> bool:
        """Attempt to sell `quantity` units of `product` (shorts allowed)."""
        timestamp = self._get_timestamp(product)
        price = self._get_price(product)
        proceeds = price * quantity

        new_cash = self.cash + proceeds
        new_positions = self.positions.copy()
        new_positions[product] = new_positions.get(product, 0) - quantity

        if not self._check_leverage(new_cash, new_positions):
            #logger.warning("Trade rejected: leverage limit exceeded.")
            return False

        self.cash = new_cash
        self.positions = new_positions
        #logger.info(f"{timestamp} | SOLD {quantity} {product} @ {price} | new cash={self.cash:.2f}")
        return True

    def summary(self) -> dict:
        """Return a snapshot of the portfolio."""
        return {
            "cash": self.cash,
            "positions": self.positions,
            "gross_exposure": self._gross_exposure(),
            "net_value": self._net_asset_value(),
            "leverage": self._leverage(),
        }

    def __str__(self) -> str:
        return str(self.summary())

In [1]:
class Market():
    def __init__(self: "Market", universe: list[str]) -> None:
        self.universe: list[str] = universe
        self.quotes: dict[str, dict] = {}  # {key: product, value: {key: timestep, value: price}}
        
    def update(self: "Market", quote: dict)-> None:
        if quote['id'] != "Clock":
            self.quotes[quote['id']] = quote

    def __str__(self: "Market") -> str:
        return str(self.quotes)

In [ ]:
import pandas as pd

data = pd.read_csv("../data/sp500_close.csv")



In [ ]:
tickers = data["Ticker"].unique()
market = Market(universe = list(tickers)) #: it is not clear what the universe does...
for ticker in tickers: 
    ticker_df = data[data["Ticker"] == ticker]
    ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)



C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)
C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)
C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-v

            Date  Price Close
0     2000-01-03    145.43750
1     2000-01-04    139.75000
2     2000-01-05    140.21875
3     2000-01-06    139.56250
4     2000-01-07    144.96875
...          ...          ...
6418  2025-07-17    628.04000
6419  2025-07-18    627.58000
6420  2025-07-18    627.58000
6421  2025-07-18    627.58000
6422  2025-07-18    627.58000

[6423 rows x 2 columns]
             Date  Price Close
6423   2000-01-03      45.0000
6424   2000-01-04      43.1250
6425   2000-01-05      43.0000
6426   2000-01-06      44.2500
6427   2000-01-07      44.1875
...           ...          ...
12841  2025-07-16      61.1400
12842  2025-07-17      62.4200
12843  2025-07-18      65.3200
12844  2025-07-18      65.3200
12845  2025-07-18      65.3200

[6423 rows x 2 columns]
             Date  Price Close
12846  2000-01-03      55.5000
12847  2000-01-04      52.8125
12848  2000-01-05      52.7500
12849  2000-01-06      53.5000
12850  2000-01-07      53.6250
...           ...          ...
1

C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)
C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)
C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-v

             Date  Price Close
19269  2002-05-03    28.786654
19270  2002-05-06    28.673257
19271  2002-05-07    28.810953
19272  2002-05-08    28.859552
19273  2002-05-09    28.584159
...           ...          ...
25103  2025-07-16    24.080000
25104  2025-07-17    24.510000
25105  2025-07-18    26.010000
25106  2025-07-18    26.010000
25107  2025-07-18    26.010000

[5839 rows x 2 columns]
             Date  Price Close
25108  2000-01-03     19.56250
25109  2000-01-04     18.46875
25110  2000-01-05     18.40625
25111  2000-01-06     19.21875
25112  2000-01-07     19.56250
...           ...          ...
31526  2025-07-16     79.91000
31527  2025-07-17     79.71000
31528  2025-07-18     80.64000
31529  2025-07-18     80.64000
31530  2025-07-18     80.64000

[6423 rows x 2 columns]
             Date  Price Close
31531  2000-01-03    46.583336
31532  2000-01-04    44.625002
31533  2000-01-05    43.958336
31534  2000-01-06    45.500002
31535  2000-01-07    45.208336
...           ...   

C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)
C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)


...

              Date  Price Close
193396  2000-01-03    26.666668
193397  2000-01-04    24.666668
193398  2000-01-05    26.125001
193399  2000-01-06    27.083335
193400  2000-01-07    27.625001
...            ...          ...
199814  2025-07-16   142.250000
199815  2025-07-17   143.290000
199816  2025-07-18   143.460000
199817  2025-07-18   143.460000
199818  2025-07-18   143.460000

[6423 rows x 2 columns]
              Date  Price Close
199819  2013-12-09        24.60
199820  2013-12-10        24.88
199821  2013-12-11        25.99
199822  2013-12-12        25.45
199823  2013-12-13        26.23
...            ...          ...
202736  2025-07-16        12.27
202737  2025-07-17        12.45
202738  2025-07-18        12.51
202739  2025-07-18        12.51
202740  2025-07-18        12.51

[2922 rows x 2 columns]
              Date  Price Close
202741  2000-01-03      61.8125
202742  2000-01-04      59.9375
202743  2000-01-05      58.6250
202744  2000-01-06      55.7500
202745  2000-01-07    

C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)
C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)
C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-v

              Date  Price Close
209164  2000-01-03     3.022496
209165  2000-01-04     3.447535
209166  2000-01-05     3.353082
209167  2000-01-06     3.447535
209168  2000-01-07     3.967026
...            ...          ...
215582  2025-07-16    66.530000
215583  2025-07-17    67.020000
215584  2025-07-18    68.000000
215585  2025-07-18    68.000000
215586  2025-07-18    68.000000

[6423 rows x 2 columns]
              Date  Price Close
215587  2000-01-03    18.791668
215588  2000-01-04    18.750001
215589  2000-01-05    19.041668
215590  2000-01-06    19.083334
215591  2000-01-07    18.583334
...            ...          ...
222005  2025-07-16   389.120000
222006  2025-07-17   397.900000
222007  2025-07-18   403.310000
222008  2025-07-18   403.310000
222009  2025-07-18   403.310000

[6423 rows x 2 columns]
              Date  Price Close
222010  2000-01-03     39.15625
222011  2000-01-04     38.40625
222012  2000-01-05     40.37500
222013  2000-01-06     42.68750
222014  2000-01-07    

C:\Users\benja\AppData\Local\Temp\ipykernel_26828\3115614510.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_df.drop(labels= ["Ticker"], axis =1, inplace = True)


KeyboardInterrupt: 